# preparing protein structures with prepmd

This tutorial will cover:
* Preparing protein structures from the PDB for simulation with prepmd
* Running simple molecular dynamics simulations with runmd
* Preparing and running simulations with ligands with prepmd and runmd
* Integrating EM data to refine a structure with TEMPy-ReFF

The final section is provided as an example of an integrative biology workflow. You are encouraged to do whatever you want with the resulting pdb, or to run prepmd using a structure that you are interested in instead of the provided examples.

We will use [human presequence protease](https://www.rcsb.org/structure/6XOV) (PDB ID: 6XOV) as our demonstration structure. This structure was derived from Cryogenic Electron Microscopy (cryo-EM) and contains a loop region that was too flexible to show up in the electron density map. The cell below will download the structure and open it in py3dmol:

In [ ]:
!wget https://files.rcsb.org/download/6XOV.pdb
import util
import py3Dmol
with open("6XOV.pdb", "r") as file:
  system = "".join([x for x in file])
view = py3Dmol.view(width=800, height=600)
view.addModel(system, "pdb")
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.zoomTo()
view.show()

Unfortunately, py3dmol doesn't show a dotted line, but you can see it on the [PDB entry](https://www.rcsb.org/structure/6XOV).

To prepare the system, we run prepmd, which will automatically work through the following steps:
* Fill missing loops
* Add missing atoms
* Fix non-standard residues
* Resolve steric clashes and minimise structure
* Remove/manage heteroatoms

By default, prepmd uses PDBFixer to fill loops, but PDBFixer's loop-filling tools are very basic. If you have a valid MODELLER license, you can use MODELLER instead by running this command:

In [ ]:
!add_modeller_license <LICENSE_KEY>

Note: if you have a local structure file you want to use, run with the `-s`\ `--structure` argument to supply it. You still need to give it an ID in place of 6XOV - this could be anything, it's just required because some tools need to see a pdb file have a pdb ID.

To prepare the structure, run the following command:

In [ ]:
!prepmd 6xov 6xov.pdb

prepmd has lots of optional features and settings, so if have more specific requirements you might want to consult the [readthedocs page](https://prepmd.readthedocs.io/en/latest/Prep-Structure.html).

The output molecule has been written to 6xov.pdb, and you might want to look at it in pymol to confirm that the loop has been filled. For now, we may as well run an MD simulation, because that's what you always do in tutorials like this.

# Run an MD simulation

prepmd has the ability to run molecular dynamics simulations by wrapping OpenMM functions. This can be useful to quickly run MD simulations for validation or as part of some larger automation, but it only supports a limited subset of OpenMM's functionality.

Here are some of the default parameters:
*   Force field: charmm36 (or amber14 if the system contains ligands)
*   Solvent: implicit equivalent to amber's igb=8 (or tip3p if the system contains ligands)
*   Timestep: 0.002 picoseconds (or 0.0002 picoseconds if the system contains ligands)
*   Constrain: Hbonds
*   Non bonded method: CutoffNonPeriodic (or PME if solvent is explicit)

runmd supports a selection of common MD settings, including several force fields,  integrators, solvent models, etc. Fore more information on these settings, [consult the runmd documentation](https://prepmd.readthedocs.io/en/latest/Run-with-runmd.html). The command below will run a simulation of 50,000 MD steps, which should run fairly quickly, and write a trajectory file in the gromacs compressed trajectory format to traj.xtc. Note: this is an extremely short simulation, if you have a GPU accessible you might want to remove the `--step` argument and change the steps to something like 50000.

In [ ]:
!runmd 6xov.pdb -o 6xov_minimised.pdb --traj_out traj.xtc --md_steps 1000 --step 10

We can now view the resulting trajectory in py3dmol:

In [ ]:
traj = util.get_pdb_traj("6xov.pdb", "traj.xtc")
view = py3Dmol.view(width=600, height=600)
view.addModelsAsFrames(traj, "pdb")
view.setStyle({"cartoon": {'color': 'spectrum'}})
view.zoomTo()
view.animate({"loop": "forward", "interval": 100})
view.show()

## Optional: run a ligand simulation
One of most fiddly and difficult things you can do with a molecular dynamics simulation is to add a ligand. prepmd can automatically use rdkit to extract ligands from structure files and fix their geometry. runmd can run simulations containing ligands, and will automatically adjust MD parameters and import the Sage force field provided by OpenFF. The cell below shows a ligand simulation of Alpha-Kinase Domain of Myosin Heavy chain Kinase (3PDT) being prepared and run. The parameter `-hetatms` tells prepmd to fix each small molecule and write it to a separate .sdf file. We're only interested in the ADP molecule, so we add it back to runmd via the `-l` argument. Note: this is a very short simulation, if you have a GPU accessible, you might want to run it on with --md_steps 10000 and no `--step` argument.

In [ ]:
!prepmd --hetatms 3PDT 3PDT.pdb
!runmd 3PDT.pdb -l ADP.sdf -o 3PDT_minimised.pdb --traj_out ligand_traj.xtc --md_steps 1000 --step 10

View the resulting trajectory:

In [ ]:
traj = util.get_pdb_traj("3PDT_minimised.pdb", "ligand_traj.xtc")
view = py3Dmol.view(width=600, height=600)
view.addModelsAsFrames(traj, "pdb")
view.setStyle({"cartoon": {'color': 'spectrum'}})
view.addStyle({"resn": "ADP"}, {"stick": {"colorscheme": "yellowCarbon"}})
view.zoomTo()
view.animate({"loop": "forward", "interval": 100})
view.show()

## Optional: refine a structure using TEMPy-ReFF
This is getting outside of the scope of the tutorial, but the whole 'make a pdb, run a simulation' thing is boring, it's what every structure preparation tutorial has you do, and although the tools and structures are much better now, it's been philosophically unchanged for a long time. There are many more interesting things you can do with a prepared .pdb file, and as a somewhat arbitrary example, this tutorial will now show you how to refine the structure you've generated so that it matches an electron density map.

The structure we initially used was derived from an cryo-EM map, but we've now made a load of modifications to it, and it no longer matches the original data. By running TEMPy-ReFF, we can fit our fixed structure back into that original map.

The code below downloads the electron density map EMD-22281 (which corresponds to the PDB ID 6XOV) from the EMDB and runs TEMPy-ReFF

In [ ]:
!wget https://files.wwpdb.org/pub/emdb/structures/EMD-22281/map/emd_22281.map.gz
!python -m tempy_reff.main --model 6xov.pdb --map emd_22281.map.gz --resolution 0.855 --fitting-gmm --output-dir tempy_out

To see how the pdb fits into the electron density map, load the isosurface and make it transparent (warning: the isosurface is very high res and will cause the viewer to consume a lot of RAM):

In [ ]:
!mv emd_22281.map.gz emd_22281.mrc.gz
util.convert_to_dx("emd_22281.mrc.gz", "emd_22281.dx")
with open("tempy_out/final.pdb", "r") as file:
  system = "".join([x for x in file])
with open("emd_22281.dx", "r") as file:
  vol = "".join([x for x in file])
view = py3Dmol.view(width=800, height=600)
view.addModel(system, "pdb")
view.setStyle({'cartoon': {'color': 'spectrum'}})
view.addVolumetricData(vol, "dx", {"isoval": 0.00538,"color": "cyan",
                                   "opacity": 0.5, "wireframe": False}
)
view.zoomTo()
view.show()

## End of tutorial!
Congratulations, you've reached the end of the tutorial! Feel free to use the remaining time to try whatever prepmd settings or molecules you like. Or do something else, I can't tell you what to do. Live is fleeting, go home, see your loved ones, etc.